# Chapter 2 — Spectral Clustering

This chapter covers two spectral graph partitioning methods:
- **Spectral Clustering** — uses the Laplacian eigenvectors + k-means
- **Normalized Cut** — minimizes a balanced cut criterion via the normalized Laplacian

Both methods exploit the eigenstructure of the graph Laplacian to find meaningful partitions.

---

## Part 1 — Initialization

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from scipy.linalg import eigh
from sklearn.cluster import KMeans

ModuleNotFoundError: No module named 'numpy'

: 

### Graph construction

We use `networkx` to generate random graphs with a fixed seed for reproducibility.  
The helper below builds the graph and computes all required matrices in one call.

In [ ]:
def build_graph_matrices(N, M, seed=42):
    """
    Build a random undirected graph and compute its core matrices.

    Parameters
    ----------
    N : int — number of nodes
    M : int — number of edges
    seed : int — random seed for reproducibility

    Returns
    -------
    G  : networkx Graph
    A  : adjacency matrix (N x N)
    D  : degree matrix (N x N diagonal)
    L  : Laplacian L = D - A
    pos: node layout for plotting
    """
    G = nx.gnm_random_graph(n=N, m=M, seed=seed)
    pos = nx.spring_layout(G, seed=seed)
    A = nx.to_numpy_array(G, dtype=float)
    degrees = A.sum(axis=1)
    D = np.diag(degrees)
    L = D - A
    return G, A, D, L, pos


def plot_graph(G, pos, labels=None, title='', cmap=plt.cm.Set1):
    """
    Draw the graph. If labels are provided, nodes are colored by cluster.
    """
    plt.figure(figsize=(6, 5))
    node_color = labels if labels is not None else 'steelblue'
    nx.draw(G, pos, with_labels=True, node_color=node_color,
            cmap=cmap, edge_color='gray', node_size=600,
            font_color='white', font_weight='bold')
    plt.title(title, fontsize=13)
    plt.tight_layout()
    plt.show()

---
## Part 2 — K-Means Algorithm

### Theory

K-Means partitions a set of points $P = \{p_i\}_{i=1}^n \subset \mathbb{R}^k$ into $k$ clusters by minimizing the total intra-cluster squared Euclidean distance:

$$\min_{C_1,\ldots,C_k} \sum_{i=1}^{k} \sum_{p \in C_i} \|p - \mu_i\|^2$$

where $\mu_i = \frac{1}{|C_i|}\sum_{p \in C_i} p$ is the centroid of cluster $i$.

The algorithm alternates between two steps until convergence:
1. **Assign** each point to the nearest centroid
2. **Update** each centroid as the mean of its assigned points

In [ ]:
def kmeans(points, k, max_iters=500, seed=0):
    """
    K-Means clustering from scratch.

    Parameters
    ----------
    points    : ndarray (n, d) — data points
    k         : int — number of clusters
    max_iters : int — maximum iterations
    seed      : int — random seed

    Returns
    -------
    centroids         : final cluster centers (k, d)
    cluster_assignments : cluster label per point (n,)
    iterations        : number of iterations until convergence
    initial_centroids : starting centroids (k, d)
    """
    np.random.seed(seed)
    indices = np.random.choice(len(points), k, replace=False)
    centroids = points[indices].copy()
    initial_centroids = centroids.copy()

    for it in range(max_iters):
        # Assign each point to the nearest centroid
        distances = np.linalg.norm(
            points[:, np.newaxis, :] - centroids[np.newaxis, :, :], axis=2
        )
        assignments = np.argmin(distances, axis=1)

        # Update centroids
        new_centroids = np.array([
            points[assignments == i].mean(axis=0)
            if np.any(assignments == i) else centroids[i]
            for i in range(k)
        ])

        if np.allclose(centroids, new_centroids):
            break
        centroids = new_centroids

    return centroids, assignments, it + 1, initial_centroids

### K-Means on random 2D points — visual example

In [ ]:
np.random.seed(42)
points = np.random.rand(300, 2)
k = 5
colors = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12', '#9b59b6']

centroids, assignments, iters, initial_centroids = kmeans(points, k)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Before clustering
axes[0].scatter(points[:, 0], points[:, 1], c='#95a5a6', alpha=0.6, s=20)
axes[0].scatter(initial_centroids[:, 0], initial_centroids[:, 1],
                c='black', marker='X', s=150, label='Initial centroids', zorder=5)
axes[0].set_title('Before clustering — initial centroids', fontsize=12)
axes[0].legend()

# After clustering
for i in range(k):
    mask = assignments == i
    axes[1].scatter(points[mask, 0], points[mask, 1],
                    c=colors[i], alpha=0.7, s=20, label=f'Cluster {i+1}')
axes[1].scatter(centroids[:, 0], centroids[:, 1],
                c='black', marker='X', s=150, zorder=5, label='Final centroids')
axes[1].set_title(f'After clustering — {iters} iterations', fontsize=12)
axes[1].legend(fontsize=8)

plt.suptitle('K-Means: 300 points, k=5', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Converged in {iters} iterations")

---
## Part 3 — Spectral Clustering Algorithm

### Theory

The key insight: the $k$ smallest eigenvectors of the Laplacian $L$ encode the cluster structure of the graph. Instead of clustering nodes directly, we embed them in $\mathbb{R}^k$ using these eigenvectors, then apply k-means.

**Algorithm:**
1. Compute adjacency matrix $A$ and Laplacian $L = D - A$
2. Compute the $k$ smallest eigenvectors of $L$ → matrix $U \in \mathbb{R}^{N \times k}$
3. Treat each row of $U$ as a point in $\mathbb{R}^k$
4. Apply k-means on these $N$ points

In [ ]:
def spectral_clustering(L, k, n_init=500):
    """
    Spectral Clustering via Laplacian eigendecomposition + k-means.

    Parameters
    ----------
    L      : Laplacian matrix (N x N)
    k      : number of clusters
    n_init : number of k-means restarts

    Returns
    -------
    labels      : cluster assignment per node (N,)
    eigenvalues : k smallest eigenvalues
    eigenvectors: k smallest eigenvectors (N x k)
    """
    eigenvalues, eigenvectors = np.linalg.eigh(L)
    top_k_vecs = eigenvectors[:, :k]
    top_k_vals = eigenvalues[:k]

    km = KMeans(n_clusters=k, n_init=n_init, random_state=0)
    labels = km.fit_predict(top_k_vecs)

    return labels, top_k_vals, top_k_vecs

### Example 1 — 8 nodes, 14 edges, k=2

In [ ]:
G1, A1, D1, L1, pos1 = build_graph_matrices(N=8, M=14, seed=42)
labels1, evals1, evecs1 = spectral_clustering(L1, k=2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

plt.sca(axes[0])
nx.draw(G1, pos1, with_labels=True, node_color='steelblue',
        edge_color='gray', node_size=600, font_color='white',
        font_weight='bold', ax=axes[0])
axes[0].set_title('Original Graph (N=8, M=14)', fontsize=12)

plt.sca(axes[1])
nx.draw(G1, pos1, with_labels=True, node_color=labels1,
        cmap=plt.cm.Set1, edge_color='gray', node_size=600,
        font_color='white', font_weight='bold', ax=axes[1])
axes[1].set_title('After Spectral Clustering (k=2)', fontsize=12)

plt.tight_layout()
plt.show()

print("Cluster assignments:", labels1)
for c in range(2):
    print(f"  Cluster {c}: nodes {list(np.where(labels1 == c)[0])}")
print("\nTop-2 eigenvalues of L:", np.round(evals1, 4))
print("Top-2 eigenvectors (rows = nodes):")
print(np.round(evecs1, 4))

### Example 2 — 12 nodes, 21 edges, k=2 and k=4

This example illustrates that k=2 can produce unbalanced partitions — a known limitation of spectral clustering.

In [ ]:
G2, A2, D2, L2, pos2 = build_graph_matrices(N=12, M=21, seed=42)

labels_k2, _, _ = spectral_clustering(L2, k=2)
labels_k4, _, _ = spectral_clustering(L2, k=4)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Original
nx.draw(G2, pos2, with_labels=True, node_color='steelblue',
        edge_color='gray', node_size=500, font_color='white',
        font_weight='bold', ax=axes[0])
axes[0].set_title('Original Graph (N=12, M=21)', fontsize=11)

# k=2
nx.draw(G2, pos2, with_labels=True, node_color=labels_k2,
        cmap=plt.cm.Set1, edge_color='gray', node_size=500,
        font_color='white', font_weight='bold', ax=axes[1])
axes[1].set_title('Spectral Clustering k=2', fontsize=11)

# k=4
nx.draw(G2, pos2, with_labels=True, node_color=labels_k4,
        cmap=plt.cm.Set1, edge_color='gray', node_size=500,
        font_color='white', font_weight='bold', ax=axes[2])
axes[2].set_title('Spectral Clustering k=4', fontsize=11)

plt.tight_layout()
plt.show()

for k_val, lbl in [(2, labels_k2), (4, labels_k4)]:
    print(f"\nk={k_val} partition:")
    for c in range(k_val):
        print(f"  Cluster {c}: nodes {list(np.where(lbl == c)[0])}")

### Eigenvalue gap — choosing k

A practical heuristic: the optimal $k$ corresponds to a significant **gap** in the sorted eigenvalue spectrum of $L$.  
The number of zero (or near-zero) eigenvalues equals the number of connected components.

In [ ]:
evals_all, _ = np.linalg.eigh(L2)

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(evals_all) + 1), evals_all, 'o-', color='steelblue', linewidth=2)
plt.axvline(x=2, color='red', linestyle='--', alpha=0.7, label='k=2 gap')
plt.axvline(x=4, color='orange', linestyle='--', alpha=0.7, label='k=4 gap')
plt.xlabel('Index', fontsize=12)
plt.ylabel('Eigenvalue', fontsize=12)
plt.title('Laplacian Eigenvalue Spectrum (N=12, M=21)', fontsize=13)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Sorted eigenvalues:", np.round(evals_all, 4))
gaps = np.diff(evals_all)
print("Eigenvalue gaps:    ", np.round(gaps, 4))
print("Largest gap at index:", np.argmax(gaps) + 2, "→ suggests k =", np.argmax(gaps) + 2)

---
## Results & Metrics

### Cut size

The **cut size** measures partition quality — total weight of edges crossing between clusters.  
Lower is better.

In [ ]:
def compute_cut_size(A, labels):
    """
    Compute the cut size: total weight of edges connecting different clusters.

    Parameters
    ----------
    A      : adjacency/weight matrix (N x N)
    labels : cluster assignment per node (N,)

    Returns
    -------
    cut : float
    """
    N = len(labels)
    cut = 0.0
    for i in range(N):
        for j in range(i + 1, N):
            if labels[i] != labels[j]:
                cut += A[i, j]
    return cut


def cluster_sizes(labels, k):
    return {c: int(np.sum(labels == c)) for c in range(k)}


# Summary table
print("=" * 50)
print("Graph: N=12, M=21")
print("=" * 50)
for k_val, lbl in [(2, labels_k2), (4, labels_k4)]:
    cut = compute_cut_size(A2, lbl)
    sizes = cluster_sizes(lbl, k_val)
    print(f"\nk = {k_val}")
    print(f"  Cut size      : {cut:.2f}")
    print(f"  Cluster sizes : {sizes}")
    balance = min(sizes.values()) / max(sizes.values())
    print(f"  Balance ratio : {balance:.2f}  (1.0 = perfectly balanced)")

### Eigenvector embedding visualization

The 2D scatter plot of the top-2 eigenvectors shows how the spectral embedding separates the nodes before k-means is applied.

In [ ]:
_, _, evecs2 = spectral_clustering(L2, k=2)

plt.figure(figsize=(6, 5))
scatter = plt.scatter(evecs2[:, 0], evecs2[:, 1],
                      c=labels_k2, cmap=plt.cm.Set1, s=120, zorder=5)
for i, (x, y) in enumerate(evecs2):
    plt.annotate(str(i), (x, y), textcoords='offset points',
                 xytext=(6, 4), fontsize=9)
plt.xlabel('1st eigenvector', fontsize=11)
plt.ylabel('2nd eigenvector', fontsize=11)
plt.title('Spectral Embedding — nodes in eigenvector space', fontsize=12)
plt.colorbar(scatter, label='Cluster')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Summary

| Property | Spectral Clustering |
|----------|--------------------|
| Input | Graph + number of clusters k |
| Core idea | Embed nodes via Laplacian eigenvectors, then cluster |
| Strength | Captures global graph structure; handles non-convex clusters |
| Weakness | Requires k upfront; can produce unbalanced partitions |
| Complexity | $O(N^3)$ eigendecomposition + $O(N \cdot k)$ k-means |

**Next:** [Chapter 3 — Normalized Cut](03_normalized_cut.ipynb)